# Disambiguation parameter optimization

> **Notebook convention:** as in [`entity_linking.ipynb`](entity_linking.ipynb), keep this notebook
> limited to function calls and variable definitions (experiment parameters, calls whose results are
> displayed). Every function, class and significant chunk of logic belongs in the support script
> [`disambiguation_optimization_utils.py`](disambiguation_optimization_utils.py), taking its
> dependencies as explicit arguments rather than reading notebook globals.

Explores the parameters of the disambiguation procedure ([`modules/geo_disambiguation.py`](modules/geo_disambiguation.py)),
scored on the **training split only** (`card_id` starting with `train_`).

**Objective**, in order of priority:
1. higher precision;
2. more correctly linked addresses;
3. fewer incorrectly linked addresses (wrong links that are not an ancestor of the true location).

**Method.** The GeoDB search does not depend on the disambiguator, so the training addresses are linked once
by the production pipeline, keeping the addresses the search returned (cached under `cache/`). Every
configuration then only replays the disambiguation of those addresses, in parallel worker processes.

**Stages.** Each stage starts from the best configuration of the previous one:
1. the weights of the weighted score;
2. the parent/child distance thresholds of the hierarchical likelihood;
3. the minimum population for the population order of magnitude;
4. the order of the disambiguation factors and their significance thresholds, by a greedy search;
5. the significance thresholds, keeping the production order (from the best of stage 3);
6. the order, by moving each factor of the production order up (from the best of stage 5).

`%autoreload` is enabled below, but the evaluation workers load the support script when they start:
rerun the cell creating the `evaluator` after editing it.

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import entity_linking_notebook_utils as utils
utils.setup_logging()
import disambiguation_optimization_utils as opt
import modules.entity_linking as entity_linking

In [2]:
SEARCH_INDEX_DIR = ".geo_db_search_index"
PARSED_ADDRESSES_CACHE_PATH = Path("experiments_data/entity_linking/parsed_addresses.jsonl")
# The training addresses as the GeoDB search returned them; rebuild after changing the search or the parsing
SEARCHED_FIELDS_CACHE_PATH = Path("cache/disambiguation_optimization/searched_train_fields.pkl")
OVERWRITE_SEARCH_CACHE = False
NUM_SEARCH_WORKERS = utils.default_num_workers()
# Each evaluation worker holds every searched training address (a few hundred MB)
NUM_EVALUATION_WORKERS = 32
# Addresses per evaluation task: small enough for a single configuration to keep every worker busy
EVALUATION_CHUNK_SIZE = 25

In [3]:
ground_truth = utils.load_ground_truth("open_data/bzkopen_linking_groundtruth.jsonl")
parsed_addresses = await utils.load_or_build_parsed_addresses(ground_truth, PARSED_ADDRESSES_CACHE_PATH)
train_rows, train_ground_truth = opt.train_split(parsed_addresses, ground_truth)
print(f"Training addresses: {len(train_rows)} of {len(ground_truth)}")

Running regex parsing on 1090 addresses...
606/1090 addresses need the LLM fallback (regex left them un/partially parsed).
Retrieving cached parsed addresses from experiments_data/entity_linking/parsed_addresses.jsonl...


Training addresses: 771 of 1090


In [4]:
searched_fields = opt.load_or_search_fields(
    train_rows, SEARCHED_FIELDS_CACHE_PATH, f"{SEARCH_INDEX_DIR}/tantivy_index", NUM_SEARCH_WORKERS,
    overwrite=OVERWRITE_SEARCH_CACHE,
)

Retrieving cached searched addresses from cache/disambiguation_optimization/searched_train_fields.pkl...


## Production configuration

The configuration `entity_linking.build_disambiguator()` builds. Replaying its disambiguation must link every
training address as the pipeline did when searching.

In [5]:
opt.cleanup_previous_run(globals()) # Allow script rerunning
production_config = opt.DisambiguationConfig.of(entity_linking.build_disambiguator())
evaluator = opt.ConfigEvaluator(
    SEARCHED_FIELDS_CACHE_PATH, train_ground_truth, NUM_EVALUATION_WORKERS, EVALUATION_CHUNK_SIZE)
opt.check_replay(evaluator, production_config, searched_fields)
evaluator.evaluate({"production": production_config})

2026-10-05 15:39:31.058 INFO     entity_linking.Disambiguator geo_disambiguation.py:359 Initialized with significance thresholds defaultdict(<function _default_significance_threshold at 0x7ff9771ff060>, {'weighted_score': 0.05, 'child_parent_likelihood': 0.05, 'fuzzy_similarity_score': 0.05, 'entity_types_matching_preferred': 0.05, 'population_order_of_magnitude': 0.05, 'country_likelihood_rank': 0.05, 'phonetic_score': 0.05, 'entity_types_matching': 0.05, 'is_preferred_name': 0.05, 'population_count': 0.05}), primary priority ['weighted_score', 'child_parent_likelihood', 'fuzzy_similarity_score', 'entity_types_matching_preferred', 'population_order_of_magnitude', 'country_likelihood_rank', 'phonetic_score'], secondary priority ['entity_types_matching', 'is_preferred_name', 'population_count'], population rounding factor 1, min population order of magnitude 500000, score prune thresholds {'fuzzy_similarity_score': 0.4, 'child_parent_likelihood': 0.4}, geo db geo.duckdb, geometry parent

Evaluating 1 configurations:   0%|          | 0/31 [00:00<?, ?it/s]

Replaying the disambiguation links all 771 addresses as the pipeline did


,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
production,0.908969,679,55,13,7,0.947662,0.989796


In [6]:
print(opt.describe_config(production_config))

# modules/geo_disambiguation.py
DISAMBIGUATION_FACTOR_PRIMARY_PRIORITY = [
    'weighted_score',
    'child_parent_likelihood',
    'fuzzy_similarity_score',
    'entity_types_matching_preferred',
    'population_order_of_magnitude',
    'country_likelihood_rank',
    'phonetic_score',
]
DISAMBIGUATION_FACTOR_SECONDARY_PRIORITY = [
    'entity_types_matching',
    'is_preferred_name',
    'population_count',
]
WEIGHTED_DISAMBIGUATION_FACTORS = {
    'child_parent_likelihood': 2,
    'phonetic_score': 2,
    'fuzzy_similarity_score': 2,
    'entity_types_matching_preferred': 1,
}
PARENT_CHILD_MAX_DISTANCE_KM_BY_ADMIN_LEVEL = {
    0: 300,
    1: 300,
    2: 100,
    3: 40,
    4: 20,
    5: 15,
}
# Disambiguator(min_population_order_of_magnitude=...)
min_population_order_of_magnitude = 500_000
# modules/entity_linking.py
SIGNIFICANCE_THRESHOLDS = {
    'weighted_score': 0.05,
    'child_parent_likelihood': 0.05,
    'fuzzy_similarity_score': 0.05,
    'entity_types_matching_preferred': 

## Stage 1: weights of the weighted score

The same four factors as the production weights. Round 1 sets every weight to 1 and tries a weight of 2 for each
factor in turn. Round 2 gives the factor that did best at 2 a weight of 3, and tries the other factors at 2 except
one, which weighs 1 or is excluded in turn; plus the other factors all at 2, and all at 1. The production weights
and all weights at 1 are evaluated as well. The best configuration of round 2 is the best overall, since round 2
includes the best of round 1.

In [7]:
weights_round_1, weights_round_2 = opt.optimize_score_weights(evaluator, production_config)
weights_round_1.evaluations

Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

Evaluating 8 configurations:   0%|          | 0/248 [00:00<?, ?it/s]

Best factor weighing 2 in round 1: fuzzy_similarity_score


,child_parent_likelihood,phonetic_score,fuzzy_similarity_score,entity_types_matching_preferred,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
"child_parent_likelihood=1, phonetic_score=1, fuzzy_similarity_score=2, entity_types_matching_preferred=1",1,1,2,1,0.908969,679,54,14,7,0.947662,0.989796
current,2,2,2,1,0.908969,679,55,13,7,0.947662,0.989796
"child_parent_likelihood=1, phonetic_score=2, fuzzy_similarity_score=1, entity_types_matching_preferred=1",1,2,1,1,0.908969,679,55,13,7,0.947662,0.989796
"child_parent_likelihood=1, phonetic_score=1, fuzzy_similarity_score=1, entity_types_matching_preferred=1",1,1,1,1,0.907631,678,55,14,7,0.946927,0.989781
"child_parent_likelihood=1, phonetic_score=1, fuzzy_similarity_score=1, entity_types_matching_preferred=2",1,1,1,2,0.907631,678,55,14,7,0.946927,0.989781
"child_parent_likelihood=2, phonetic_score=1, fuzzy_similarity_score=1, entity_types_matching_preferred=1",2,1,1,1,0.902276,674,60,13,7,0.943978,0.989721


In [8]:
weights_round_2.evaluations

,child_parent_likelihood,phonetic_score,fuzzy_similarity_score,entity_types_matching_preferred,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
"child_parent_likelihood=0, phonetic_score=2, fuzzy_similarity_score=3, entity_types_matching_preferred=2",0,2,3,2,0.911647,681,52,14,7,0.949129,0.989826
"child_parent_likelihood=1, phonetic_score=2, fuzzy_similarity_score=3, entity_types_matching_preferred=2",1,2,3,2,0.910308,680,53,14,7,0.948396,0.989811
"child_parent_likelihood=2, phonetic_score=2, fuzzy_similarity_score=3, entity_types_matching_preferred=0",2,2,3,0,0.910308,680,54,13,7,0.948396,0.989811
"child_parent_likelihood=1, phonetic_score=1, fuzzy_similarity_score=2, entity_types_matching_preferred=1",1,1,2,1,0.908969,679,54,14,7,0.947662,0.989796
"child_parent_likelihood=2, phonetic_score=2, fuzzy_similarity_score=3, entity_types_matching_preferred=2",2,2,3,2,0.908969,679,54,14,7,0.947662,0.989796
"child_parent_likelihood=1, phonetic_score=1, fuzzy_similarity_score=3, entity_types_matching_preferred=1",1,1,3,1,0.908969,679,54,14,7,0.947662,0.989796
"child_parent_likelihood=2, phonetic_score=2, fuzzy_similarity_score=3, entity_types_matching_preferred=1",2,2,3,1,0.908969,679,55,13,7,0.947662,0.989796
"child_parent_likelihood=2, phonetic_score=1, fuzzy_similarity_score=3, entity_types_matching_preferred=2",2,1,3,2,0.907631,678,55,14,7,0.946927,0.989781
"child_parent_likelihood=1, phonetic_score=1, fuzzy_similarity_score=1, entity_types_matching_preferred=1",1,1,1,1,0.907631,678,55,14,7,0.946927,0.989781
"child_parent_likelihood=2, phonetic_score=0, fuzzy_similarity_score=3, entity_types_matching_preferred=2",2,0,3,2,0.906292,677,56,14,7,0.946191,0.989766


In [9]:
after_weights = weights_round_2.best_config
print(f"Best weights: {after_weights.score_weights}")

Best weights: {'phonetic_score': 2, 'fuzzy_similarity_score': 3, 'entity_types_matching_preferred': 2}


## Stage 2: distance thresholds of the hierarchical likelihood

Distance from a parent beyond which a child is considered outside it, by the parent's admin level
(`PARENT_CHILD_MAX_DISTANCE_KM_BY_ADMIN_LEVEL`). The production cascade is scaled so that the first threshold
(a region or a state) is each of the distances below, keeping the ratios between levels.

In [10]:
FIRST_DISTANCES_KM = [100, 200, 300, 400, 500]
distances = opt.optimize_parent_child_distances(evaluator, after_weights, FIRST_DISTANCES_KM)
distances.evaluations

Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

,admin level 0 (km),admin level 1 (km),admin level 2 (km),admin level 3 (km),admin level 4 (km),admin level 5 (km),precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
300 / 300 / 100 / 40 / 20 / 15,300,300,100,40,20,15,0.911647,681,52,14,7,0.949129,0.989826
200 / 200 / 67 / 27 / 13 / 10,200,200,67,27,13,10,0.910308,680,53,14,7,0.948396,0.989811
400 / 400 / 133 / 53 / 27 / 20,400,400,133,53,27,20,0.908969,679,55,13,7,0.947662,0.989796
100 / 100 / 33 / 13 / 7 / 5,100,100,33,13,7,5,0.907631,678,54,15,7,0.946927,0.989781
500 / 500 / 167 / 67 / 33 / 25,500,500,167,67,33,25,0.907631,678,56,13,7,0.946927,0.989781


In [11]:
after_distances = distances.best_config
print(f"Best distances: {after_distances.parent_child_max_distance_km_by_admin_level}")

Best distances: {0: 300, 1: 300, 2: 100, 3: 40, 4: 20, 5: 15}


## Stage 3: minimum population for the population order of magnitude

Population under which `population_order_of_magnitude` is 0.

In [12]:
MIN_POPULATIONS = [0, 100, 1_000, 10_000, 100_000, 500_000, 1_000_000]
population = opt.optimize_min_population_order_of_magnitude(evaluator, after_distances, MIN_POPULATIONS)
population.evaluations

Evaluating 6 configurations:   0%|          | 0/186 [00:00<?, ?it/s]

,min_population_order_of_magnitude,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
100_000,100000,0.915663,684,52,11,7,0.951321,0.989870
1_000_000,1000000,0.912985,682,51,14,7,0.949861,0.989840
10_000,10000,0.911765,682,57,9,6,0.949861,0.991279
1_000,1000,0.911765,682,58,8,6,0.949861,0.991279
500_000,500000,0.911647,681,52,14,7,0.949129,0.989826
100,100,0.907754,679,62,7,6,0.947662,0.991241
0,0,0.906417,678,63,7,6,0.946927,0.991228


In [13]:
after_population = population.best_config
print(f"Best min population: {after_population.min_population_order_of_magnitude:_}")

Best min population: 100_000


## Stage 4: order of the factors, by a greedy search

Which factor alone decides best? With that one set, which factor decides best after it, and with which
significance threshold on the first? This repeats until all 10 factors are ordered, and a final step picks
the threshold of the last factor. The factor just added gets a threshold of 0 until the next step sets it.

The ordered factors are all primary factors: candidates no factor tells apart by its threshold are compared
again on the same factors with no threshold (see `geo_disambiguation.comparison_steps`); there are no
secondary factors.

Each factor gets 5 threshold candidates suited to the range of its address-level score (an average over the
address' entities):
- scores between 0 and 1 (`weighted_score`, `fuzzy_similarity_score`, `phonetic_score`): fine steps around the production 0.05;
- `child_parent_likelihood` and the entity type matches: coarser steps, since per-entity values are few
  (e.g. 0, 0.25, 0.75, 1) or are ratios of admin codes;
- `is_preferred_name`: 0 or 1 per entity;
- `country_likelihood_rank`: 0 to 3 per entity;
- `population_order_of_magnitude`: 0 or the rounded log10 of the population (about 2 to 7);
- `population_count`: the raw population (`DISAMBIGUATION_POPULATION_ROUNDING_FACTOR` is 1).

In [14]:
SIGNIFICANCE_THRESHOLD_CANDIDATES = {
    # in the production priority order, which breaks ties between orders
    "weighted_score": [0, 0.025, 0.05, 0.1, 0.2],
    "child_parent_likelihood": [0, 0.05, 0.1, 0.2, 0.3],
    "fuzzy_similarity_score": [0, 0.025, 0.05, 0.1, 0.2],
    "entity_types_matching_preferred": [0, 0.05, 0.1, 0.2, 0.3],
    "population_order_of_magnitude": [0, 0.25, 0.5, 1, 2],
    "country_likelihood_rank": [0, 0.25, 0.5, 1, 1.5],
    "phonetic_score": [0, 0.025, 0.05, 0.1, 0.2],
    "entity_types_matching": [0, 0.05, 0.1, 0.2, 0.3],
    "is_preferred_name": [0, 0.1, 0.25, 0.5, 0.75],
    "population_count": [0, 1_000, 10_000, 100_000, 1_000_000],
}

In [15]:
order_search = opt.greedy_order_search(evaluator, after_population, SIGNIFICANCE_THRESHOLD_CANDIDATES)
order_search.summary

Evaluating 10 configurations:   0%|          | 0/310 [00:00<?, ?it/s]

Step: weighted_score (0)


Evaluating 45 configurations:   0%|          | 0/1395 [00:00<?, ?it/s]

Step: weighted_score (0) > population_count (0)


Evaluating 40 configurations:   0%|          | 0/1240 [00:00<?, ?it/s]

...

Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

Step: weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0) > fuzzy_similarity_score (0)


Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

Step: weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0) > fuzzy_similarity_score (0)
Best order (step 9): weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0)


,added factor,threshold set on,threshold,configurations evaluated,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
step,,,,,,,,,,,
1,weighted_score,NaN,NaN,10,0.622378,356,42,174,182,0.641441,0.661710
2,population_count,weighted_score,0.0,45,0.842953,628,104,13,9,0.908828,0.985871
3,is_preferred_name,population_count,100000.0,40,0.867114,646,86,13,9,0.922857,0.986260
4,child_parent_likelihood,is_preferred_name,0.0,35,0.880537,656,80,9,9,0.930496,0.986466
5,country_likelihood_rank,child_parent_likelihood,0.0,30,0.883534,660,72,15,7,0.933522,0.989505
6,phonetic_score,country_likelihood_rank,0.0,25,0.886212,662,70,15,7,0.935028,0.989537
7,entity_types_matching_preferred,phonetic_score,0.0,20,0.886212,662,70,15,7,0.935028,0.989537
8,population_order_of_magnitude,entity_types_matching_preferred,0.0,15,0.886212,662,70,15,7,0.935028,0.989537
9,entity_types_matching,population_order_of_magnitude,0.0,10,0.886212,662,70,15,7,0.935028,0.989537


In [16]:
opt.display_greedy_steps(order_search, top=5)

**Step 1**: factor added after nan, and the threshold of the latter

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
weighted_score (0),weighted_score,None,0.622378,356,42,174,182,0.641441,0.661710
population_count (0),population_count,None,0.591700,442,237,68,7,0.739130,0.984410
fuzzy_similarity_score (0),fuzzy_similarity_score,None,0.567857,318,31,211,194,0.593284,0.621094
population_order_of_magnitude (0),population_order_of_magnitude,None,0.552817,314,77,177,186,0.588015,0.628000
phonetic_score (0),phonetic_score,None,0.541667,286,31,211,226,0.550000,0.558594


**Step 2**: factor added after weighted_score, and the threshold of the latter

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
weighted_score (0) > population_count (0),population_count,0.000,0.842953,628,104,13,9,0.908828,0.985871
weighted_score (0.025) > population_count (0),population_count,0.025,0.836461,624,112,10,8,0.905660,0.987342
weighted_score (0.05) > population_count (0),population_count,0.050,0.827309,618,122,7,7,0.900875,0.988800
weighted_score (0.1) > population_count (0),population_count,0.100,0.795181,594,146,7,7,0.881306,0.988353
weighted_score (0.05) > population_order_of_magnitude (0),population_order_of_magnitude,0.050,0.773826,544,50,109,51,0.838213,0.914286


**Step 3**: factor added after population_count, and the threshold of the latter

...

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0),entity_types_matching,0.00,0.886212,662,70,15,7,0.935028,0.989537
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0.25) > entity_types_matching (0),entity_types_matching,0.25,0.886212,662,70,15,7,0.935028,0.989537
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0.5) > entity_types_matching (0),entity_types_matching,0.50,0.886212,662,70,15,7,0.935028,0.989537
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (1) > entity_types_matching (0),entity_types_matching,1.00,0.886212,662,70,15,7,0.935028,0.989537
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (2) > entity_types_matching (0),entity_types_matching,2.00,0.886212,662,70,15,7,0.935028,0.989537


**Step 10**: factor added after entity_types_matching, and the threshold of the latter

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0) > fuzzy_similarity_score (0),fuzzy_similarity_score,0.00,0.884873,661,71,15,7,0.934276,0.989521
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0.05) > fuzzy_similarity_score (0),fuzzy_similarity_score,0.05,0.884873,661,71,15,7,0.934276,0.989521
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0.1) > fuzzy_similarity_score (0),fuzzy_similarity_score,0.10,0.884873,661,71,15,7,0.934276,0.989521
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0.2) > fuzzy_similarity_score (0),fuzzy_similarity_score,0.20,0.884873,661,71,15,7,0.934276,0.989521
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0.3) > fuzzy_similarity_score (0),fuzzy_similarity_score,0.30,0.884873,661,71,15,7,0.934276,0.989521


**Step 11**: factor added after fuzzy_similarity_score, and the threshold of the latter

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0) > fuzzy_similarity_score (0),None,0.000,0.884873,661,71,15,7,0.934276,0.989521
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0) > fuzzy_similarity_score (0.025),None,0.025,0.884873,661,71,15,7,0.934276,0.989521
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0) > fuzzy_similarity_score (0.05),None,0.050,0.884873,661,71,15,7,0.934276,0.989521
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0) > fuzzy_similarity_score (0.1),None,0.100,0.884873,661,71,15,7,0.934276,0.989521
weighted_score (0) > population_count (100000) > is_preferred_name (0) > child_parent_likelihood (0) > country_likelihood_rank (0) > phonetic_score (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > entity_types_matching (0) > fuzzy_similarity_score (0.2),None,0.200,0.884873,661,71,15,7,0.934276,0.989521


## Stage 5: significance thresholds, in the production order

Keeps the production order of the factors (primary, then secondary) and the best configuration of stage 3. Greedily,
from the first factor to the last, each factor tries the 5 threshold candidates of stage 4, with the thresholds
already chosen for the factors before it. The factor's current threshold (0.05 in production) is tried too, and
kept among ties.

In [17]:
threshold_search = opt.optimize_significance_thresholds(evaluator, after_population, SIGNIFICANCE_THRESHOLD_CANDIDATES)
threshold_search.summary

Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

weighted_score: 0.05 -> 0.05


Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

child_parent_likelihood: 0.05 -> 0.05


Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

...

Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

is_preferred_name: 0.05 -> 0.5


Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

population_count: 0.05 -> 0.05


,previous threshold,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
factor,,,,,,,,,
weighted_score,0.05,0.05,0.915663,684,52,11,7,0.951321,0.989870
child_parent_likelihood,0.05,0.05,0.915663,684,52,11,7,0.951321,0.989870
fuzzy_similarity_score,0.05,0.05,0.915663,684,52,11,7,0.951321,0.989870
entity_types_matching_preferred,0.05,0.05,0.915663,684,52,11,7,0.951321,0.989870
population_order_of_magnitude,0.05,0.05,0.915663,684,52,11,7,0.951321,0.989870
country_likelihood_rank,0.05,0.50,0.917001,685,51,11,7,0.952050,0.989884
phonetic_score,0.05,0.05,0.917001,685,51,11,7,0.952050,0.989884
entity_types_matching,0.05,0.05,0.917001,685,51,11,7,0.952050,0.989884
is_preferred_name,0.05,0.50,0.918231,685,51,10,8,0.952050,0.988456


In [18]:
opt.display_sequential_steps(threshold_search)

**weighted_score**

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
weighted_score = 0.05,0.050,True,0.915663,684,52,11,7,0.951321,0.989870
weighted_score = 0.1,0.100,False,0.914324,683,53,11,7,0.950592,0.989855
weighted_score = 0,0.000,False,0.911647,681,54,12,7,0.949129,0.989826
weighted_score = 0.025,0.025,False,0.911647,681,55,11,7,0.949129,0.989826
weighted_score = 0.2,0.200,False,0.908969,679,57,11,7,0.947662,0.989796


**child_parent_likelihood**

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
child_parent_likelihood = 0.05,0.05,True,0.915663,684,52,11,7,0.951321,0.989870
child_parent_likelihood = 0.1,0.10,False,0.915663,684,52,11,7,0.951321,0.989870
child_parent_likelihood = 0,0.00,False,0.911647,681,55,11,7,0.949129,0.989826
child_parent_likelihood = 0.2,0.20,False,0.911647,681,55,11,7,0.949129,0.989826
child_parent_likelihood = 0.3,0.30,False,0.911647,681,55,11,7,0.949129,0.989826


**fuzzy_similarity_score**

...

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
entity_types_matching = 0.05,0.05,True,0.917001,685,51,11,7,0.95205,0.989884
entity_types_matching = 0,0.00,False,0.917001,685,51,11,7,0.95205,0.989884
entity_types_matching = 0.1,0.10,False,0.917001,685,51,11,7,0.95205,0.989884
entity_types_matching = 0.2,0.20,False,0.917001,685,51,11,7,0.95205,0.989884
entity_types_matching = 0.3,0.30,False,0.917001,685,51,11,7,0.95205,0.989884


**is_preferred_name**

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
is_preferred_name = 0.5,0.50,False,0.918231,685,51,10,8,0.95205,0.988456
is_preferred_name = 0.75,0.75,False,0.918231,685,51,10,8,0.95205,0.988456
is_preferred_name = 0.05,0.05,True,0.917001,685,51,11,7,0.95205,0.989884
is_preferred_name = 0,0.00,False,0.917001,685,51,11,7,0.95205,0.989884
is_preferred_name = 0.1,0.10,False,0.917001,685,51,11,7,0.95205,0.989884
is_preferred_name = 0.25,0.25,False,0.917001,685,51,11,7,0.95205,0.989884


**population_count**

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
population_count = 0.05,0.05,True,0.918231,685,51,10,8,0.952050,0.988456
population_count = 0,0.00,False,0.918231,685,51,10,8,0.952050,0.988456
population_count = 1000,1000.00,False,0.902145,673,50,23,8,0.943238,0.988253
population_count = 10000,10000.00,False,0.888740,663,47,36,8,0.935780,0.988077
population_count = 100000,100000.00,False,0.858871,639,46,59,10,0.917444,0.984592
population_count = 1e+06,1000000.00,False,0.858871,639,46,59,10,0.917444,0.984592


In [19]:
after_thresholds = threshold_search.best_config

## Stage 6: order, by moving factors up

Starts from the best configuration of stage 5. From the last factor of the production order to the first, each
factor moves up one position at a time while the objective is not worse than at the position before (ties keep
moving it up). It stays at the last position that was not worse. The boundary between primary and secondary
factors stays at the same position: a secondary factor moved above it becomes primary, and the last primary factor
becomes secondary.

In [20]:
order_mutation = opt.move_factors_up(evaluator, after_thresholds)
order_mutation.summary

Evaluating 9 configurations:   0%|          | 0/279 [00:00<?, ?it/s]

population_count: position 10 -> 10


Evaluating 8 configurations:   0%|          | 0/248 [00:00<?, ?it/s]

is_preferred_name: position 9 -> 8


Evaluating 7 configurations:   0%|          | 0/217 [00:00<?, ?it/s]

...

Evaluating 3 configurations:   0%|          | 0/93 [00:00<?, ?it/s]

fuzzy_similarity_score: position 4 -> 4


Evaluating 1 configurations:   0%|          | 0/31 [00:00<?, ?it/s]

child_parent_likelihood: position 3 -> 2
weighted_score: position 1 -> 1


,from position,to position,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
factor,,,,,,,,,
population_count,10,10,0.918231,685.0,51.0,10.0,8.0,0.952050,0.988456
is_preferred_name,9,8,0.918231,685.0,51.0,10.0,8.0,0.952050,0.988456
entity_types_matching,9,7,0.919571,686.0,50.0,10.0,8.0,0.952778,0.988473
phonetic_score,8,8,0.919571,686.0,50.0,10.0,8.0,0.952778,0.988473
country_likelihood_rank,6,6,0.919571,686.0,50.0,10.0,8.0,0.952778,0.988473
population_order_of_magnitude,5,4,0.919571,686.0,50.0,10.0,8.0,0.952778,0.988473
entity_types_matching_preferred,5,2,0.919571,686.0,50.0,10.0,8.0,0.952778,0.988473
fuzzy_similarity_score,4,4,0.919571,686.0,50.0,10.0,8.0,0.952778,0.988473
child_parent_likelihood,3,2,0.919571,686.0,50.0,10.0,8.0,0.952778,0.988473


In [21]:
opt.display_sequential_steps(order_mutation)

**population_count**

,position,worse than the position below,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
population_count at position 9,9,True,0.914209,682,55,9,8,0.949861,0.988406
population_count at position 8,8,False,0.890228,665,76,6,7,0.937280,0.989583
population_count at position 7,7,False,0.885027,662,81,5,6,0.935028,0.991018
population_count at position 6,6,False,0.875668,655,88,5,6,0.929737,0.990923
population_count at position 5,5,False,0.875668,655,88,5,6,0.929737,0.990923
population_count at position 4,4,False,0.875668,655,88,5,6,0.929737,0.990923
population_count at position 3,3,False,0.863636,646,97,5,6,0.922857,0.990798
population_count at position 2,2,False,0.842246,630,113,5,6,0.910405,0.990566
population_count at position 1,1,False,0.604278,452,249,47,6,0.749585,0.986900


**is_preferred_name**

,position,worse than the position below,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
is_preferred_name at position 8,8,False,0.918231,685,51,10,8,0.952050,0.988456
is_preferred_name at position 7,7,True,0.914324,683,53,11,7,0.950592,0.989855
is_preferred_name at position 6,6,False,0.911647,681,55,11,7,0.949129,0.989826
is_preferred_name at position 5,5,False,0.903485,674,61,11,8,0.943978,0.988270
is_preferred_name at position 4,4,False,0.903485,674,61,11,8,0.943978,0.988270
is_preferred_name at position 3,3,False,0.887399,662,70,14,8,0.935028,0.988060
is_preferred_name at position 2,2,False,0.887399,662,70,14,8,0.935028,0.988060
is_preferred_name at position 1,1,False,0.867114,646,83,16,9,0.922857,0.986260


**entity_types_matching**

...

,position,worse than the position below,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
fuzzy_similarity_score at position 3,3,True,0.91689,684,52,10,8,0.951321,0.988439
fuzzy_similarity_score at position 2,2,False,0.91689,684,52,10,8,0.951321,0.988439
fuzzy_similarity_score at position 1,1,False,0.91834,686,51,10,7,0.952778,0.989899


**child_parent_likelihood**

,position,worse than the position below,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
child_parent_likelihood at position 2,2,False,0.919571,686,50,10,8,0.952778,0.988473
child_parent_likelihood at position 1,1,True,0.904826,675,62,9,8,0.944717,0.988287


**weighted_score**

_already first_

In [22]:
after_order_mutation = order_mutation.best_config

## Summary

In [23]:
stage_configs = {
    "production": production_config,
    "1. weights": after_weights,
    "2. distances": after_distances,
    "3. min population": after_population,
    "4. greedy factor order": order_search.best_config,
    "5. thresholds": after_thresholds,
    "6. moving factors up": after_order_mutation,
}
opt.compare_stages(evaluator, stage_configs)

,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall,Δ precision,Δ correctly_linked,Δ incorrectly_linked
production,0.908969,679,55,13,7,0.947662,0.989796,0.000000,0.0,0.0
1. weights,0.911647,681,52,14,7,0.949129,0.989826,0.002677,2.0,-3.0
2. distances,0.911647,681,52,14,7,0.949129,0.989826,0.002677,2.0,-3.0
3. min population,0.915663,684,52,11,7,0.951321,0.989870,0.006693,5.0,-3.0
4. greedy factor order,0.886212,662,70,15,7,0.935028,0.989537,-0.022758,-17.0,15.0
5. thresholds,0.918231,685,51,10,8,0.952050,0.988456,0.009261,6.0,-4.0
6. moving factors up,0.919571,686,50,10,8,0.952778,0.988473,0.010602,7.0,-5.0


In [24]:
opt.deciding_factor_counts(evaluator, stage_configs)

,production,1. weights,2. distances,3. min population,4. greedy factor order,5. thresholds,6. moving factors up
(common parent of tied candidates),13,14,14,9,13,8,9
(single candidate),89,89,89,89,89,89,89
child_parent_likelihood,11,54,54,54,33,54,54
child_parent_likelihood (no threshold),0,7,7,6,0,6,4
country_likelihood_rank,46,46,46,25,20,24,24
country_likelihood_rank (no threshold),0,0,0,0,0,1,1
entity_types_matching,32,31,31,28,2,28,27
entity_types_matching_preferred,3,0,0,0,0,0,0
fuzzy_similarity_score,45,24,24,24,0,24,24
is_preferred_name,10,10,10,3,37,1,1


In [25]:
print(opt.describe_config(after_order_mutation))

# modules/geo_disambiguation.py
DISAMBIGUATION_FACTOR_PRIMARY_PRIORITY = [
    'weighted_score',
    'child_parent_likelihood',
    'entity_types_matching_preferred',
    'fuzzy_similarity_score',
    'population_order_of_magnitude',
    'country_likelihood_rank',
    'entity_types_matching',
]
DISAMBIGUATION_FACTOR_SECONDARY_PRIORITY = [
    'phonetic_score',
    'is_preferred_name',
    'population_count',
]
WEIGHTED_DISAMBIGUATION_FACTORS = {
    'phonetic_score': 2,
    'fuzzy_similarity_score': 3,
    'entity_types_matching_preferred': 2,
}
PARENT_CHILD_MAX_DISTANCE_KM_BY_ADMIN_LEVEL = {
    0: 300,
    1: 300,
    2: 100,
    3: 40,
    4: 20,
    5: 15,
}
# Disambiguator(min_population_order_of_magnitude=...)
min_population_order_of_magnitude = 100_000
# modules/entity_linking.py
SIGNIFICANCE_THRESHOLDS = {
    'weighted_score': 0.05,
    'child_parent_likelihood': 0.05,
    'fuzzy_similarity_score': 0.05,
    'entity_types_matching_preferred': 0.05,
    'population_order_of_mag

### Findings (run of 2026-10-05, 771 training addresses)

Objective: precision, then correctly linked, then incorrectly linked.

| Stage | Precision | Correctly linked | Incorrectly linked | Partially linked | Failed to link | F1 |
|---|---|---|---|---|---|---|
| production | 0.9090 | 679 | 55 | 13 | 7 | 0.9477 |
| 1. weights | 0.9116 | 681 | 52 | 14 | 7 | 0.9491 |
| 2. distances | 0.9116 | 681 | 52 | 14 | 7 | 0.9491 |
| 3. min population | 0.9157 | 684 | 52 | 11 | 7 | 0.9513 |
| 4. greedy factor order | 0.8862 | 662 | 70 | 15 | 7 | 0.9350 |
| 5. thresholds | 0.9182 | 685 | 51 | 10 | 8 | 0.9521 |
| 6. moving factors up | **0.9196** | **686** | **50** | 10 | 8 | **0.9528** |

- **Weights:** fuzzy 3, phonetic 2, preferred entity type 2, with `child_parent_likelihood` excluded from the
  weighted score (it still takes part on its own).
- **Distances:** the production cascade (300 / 300 / 100 / 40 / 20 / 15 km) remains the best.
- **Min population:** 100 000 instead of 500 000.
- **Greedy order:** still well below the production order. Judged with nothing after it to break ties, each
  partial order favours factors that rarely tie over factors that decide well.
- **Thresholds (production order):** only two change, both to 0.5: `country_likelihood_rank` and
  `is_preferred_name`. The others keep 0.05.
- **Moving factors up:** the order becomes `weighted_score` > `child_parent_likelihood` >
  `entity_types_matching_preferred` > `fuzzy_similarity_score` > `population_order_of_magnitude` >
  `country_likelihood_rank` > `entity_types_matching` (primary), then `phonetic_score` > `is_preferred_name` >
  `population_count` (secondary). `entity_types_matching` moves up into the primary factors, pushing
  `phonetic_score` down to secondary. Only the move of `entity_types_matching` changes the training results (+1 correctly linked,
  −1 incorrectly linked); the other moves are ties.

The best configuration is stage 6, printed above.

In [26]:
evaluator.close()